# Ajuste do modelo de Riscos Proporcionais de COX

In [9]:
# Configuração inicial
rm(list = ls(all = TRUE))

library(pacman)
p_load(tidyverse, survival, smcure)

seed <- 123
set.seed(seed)

## Leitura dos dados de treino e teste

In [10]:
dados_treino <- read.csv("../../../dados/dados-processados/dados_treino.csv", stringsAsFactors = FALSE)
dados_teste <- read.csv("../../../dados/dados-processados/dados_teste.csv", stringsAsFactors = FALSE)

vars_paper <- c(
  "rhc_estadiamento_clinico",
  "instrucao",
  "rhc_primeiro_tratamento_recebido_no_hospital_hormonioterapia",
  "rhc_primeiro_tratamento_recebido_no_hospital_radioterapia",
  "macroregiao"
)

prepara_base <- function(df) {
  df %>%
    select(id_paciente, tempo, delta_t, all_of(vars_paper)) %>%
    mutate(
      id_paciente = as.numeric(id_paciente),
      tempo = as.numeric(tempo),
      delta_t = as.numeric(delta_t),
      across(all_of(vars_paper), as.factor)
    ) %>%
    filter(tempo > 0, !is.na(delta_t)) %>%
    drop_na(all_of(vars_paper))
}

dados_treino_modelo <- prepara_base(dados_treino)
dados_teste_modelo <- prepara_base(dados_teste)

## Cox simples e modelo com fração de cura

In [11]:
form_latencia <- Surv(tempo, delta_t) ~
  rhc_estadiamento_clinico +
  instrucao +
  rhc_primeiro_tratamento_recebido_no_hospital_hormonioterapia +
  rhc_primeiro_tratamento_recebido_no_hospital_radioterapia +
  macroregiao

modelo_cox <- coxph(form_latencia, data = dados_treino_modelo)

summary(modelo_cox)



Call:
coxph(formula = form_latencia, data = dados_treino_modelo)

  n= 2384, number of events= 308 

                                                                    coef
rhc_estadiamento_clinicoIII e IV                                 1.49975
instrucao1                                                      -0.26331
instrucao2                                                      -0.62721
rhc_primeiro_tratamento_recebido_no_hospital_hormonioterapiaSim -0.46186
rhc_primeiro_tratamento_recebido_no_hospital_radioterapiaSim    -0.21418
macroregiao2                                                    -1.05417
macroregiao3                                                    -0.03911
                                                                exp(coef)
rhc_estadiamento_clinicoIII e IV                                  4.48059
instrucao1                                                        0.76850
instrucao2                                                        0.53408
rhc_primeiro_tratam

In [13]:
# Modelo com fracao de cura (incidencia + latencia)
X_cura <- model.matrix(~ ., data = dados_treino_modelo[, vars_paper, drop = FALSE])
X_cura <- X_cura[, colnames(X_cura) != "(Intercept)", drop = FALSE]
X_cura <- X_cura[, apply(X_cura, 2, function(x) stats::sd(x, na.rm = TRUE) > 0), drop = FALSE]

dados_treino_cura <- data.frame(
  tempo = dados_treino_modelo$tempo,
  delta_t = as.integer(dados_treino_modelo$delta_t == 1),
  X_cura,
  check.names = TRUE
)

covs_cura <- setdiff(names(dados_treino_cura), c("tempo", "delta_t"))
form_latencia_cura <- as.formula(paste("Surv(tempo, delta_t) ~", paste(covs_cura, collapse = " + ")))
form_incidencia_cura <- as.formula(paste("~", paste(covs_cura, collapse = " + ")))

modelo_cura <- smcure(
  form_latencia_cura,
  cureform = form_incidencia_cura,
  data = dados_treino_cura,
  model = "ph",
  emmax = 1000,
  eps = 1e-06,
  Var = FALSE
)

summary(modelo_cura)

Program is running..be patient... done.
Call:
smcure(formula = form_latencia_cura, cureform = form_incidencia_cura, 
    data = dados_treino_cura, model = "ph", Var = FALSE, emmax = 1000, 
    eps = 1e-06)

Cure probability model:
                                                                  Estimate
(Intercept)                                                     -1.9197895
rhc_estadiamento_clinicoIII.e.IV                                 1.4365591
instrucao1                                                      -0.1762765
instrucao2                                                      -0.1309944
rhc_primeiro_tratamento_recebido_no_hospital_hormonioterapiaSim -0.1010006
rhc_primeiro_tratamento_recebido_no_hospital_radioterapiaSim     0.2589578
macroregiao2                                                    -1.1887959
macroregiao3                                                    -0.1212878


Failure time distribution model:
                                                           

          Length Class  Mode     
logistfit   30   glm    list     
b            8   -none- numeric  
beta         7   -none- numeric  
call         8   -none- call     
bnm          8   -none- character
betanm       7   -none- character
s         2384   -none- numeric  
Time      2384   -none- numeric  

## S(t) nos tempos de evento do teste

In [14]:
tempos_evento_teste <- sort(unique(dados_teste_modelo$tempo[dados_teste_modelo$delta_t == 1]))
tempos_evento_teste

[1]   1   2   4   7   8   9  10  12  13  14  15  16  20  21  22  24  25  26  27
[20]  28  30  31  33  34  36  37  38  39  40  41  42  44  47  48  49  51  52  56
[39]  60  61  62  63  67  69  75  78  81  82  83  84  87  90 112 117

In [15]:
# Previsao do modelo Cox para teste nos tempos_evento_teste
lp_teste <- predict(modelo_cox, newdata = dados_teste_modelo, type = "lp")
haz_base <- basehaz(modelo_cox, centered = FALSE)
H0_t <- stepfun(haz_base$time, c(0, haz_base$hazard), right = TRUE)(tempos_evento_teste)
S_mat_cox <- exp(-outer(H0_t, exp(lp_teste)))

pred_event_cox <- bind_rows(lapply(seq_len(nrow(dados_teste_modelo)), function(i) {
  data.frame(
    ID = dados_teste_modelo$id_paciente[i],
    TIME = tempos_evento_teste,
    PRED_S = as.numeric(S_mat_cox[, i]),
    stringsAsFactors = FALSE
  )
}))

dir.create("../../../ajuste-modelos/cox/dados/saida", recursive = TRUE, showWarnings = FALSE)
arquivo_pred_cox <- "../../../ajuste-modelos/cox/dados/saida/predicao_cox.csv"
write.csv(pred_event_cox, arquivo_pred_cox, row.names = FALSE)

In [16]:
# Previsao do modelo de cura para teste nos tempos_evento_teste
X_cura_teste <- model.matrix(~ ., data = dados_teste_modelo[, vars_paper, drop = FALSE])
X_cura_teste <- X_cura_teste[, colnames(X_cura_teste) != "(Intercept)", drop = FALSE]
X_cura_teste <- as.data.frame(X_cura_teste)
faltantes <- setdiff(covs_cura, names(X_cura_teste))
if (length(faltantes) > 0) for (nm in faltantes) X_cura_teste[[nm]] <- 0
X_cura_teste <- as.matrix(X_cura_teste[, covs_cura, drop = FALSE])

pred_cura_raw <- predictsmcure(
  modelo_cura,
  newX = X_cura_teste,
  newZ = X_cura_teste,
  model = "ph"
 )

pred_cura_df <- as.data.frame(pred_cura_raw$prediction)
col_tempo_cura <- if ("Time" %in% names(pred_cura_df)) "Time" else names(pred_cura_df)[ncol(pred_cura_df)]
tempos_cura <- as.numeric(pred_cura_df[[col_tempo_cura]])
S_cura_mat <- as.matrix(pred_cura_df[, setdiff(names(pred_cura_df), col_tempo_cura), drop = FALSE])

S_cura_interp <- sapply(seq_len(ncol(S_cura_mat)), function(j) {
  approx(
    x = tempos_cura,
    y = as.numeric(S_cura_mat[, j]),
    xout = tempos_evento_teste,
    method = "constant",
    f = 0,
    rule = 2,
    ties = "ordered"
  )$y
})
if (is.vector(S_cura_interp)) S_cura_interp <- matrix(S_cura_interp, ncol = 1)

pred_event_cura <- bind_rows(lapply(seq_len(ncol(S_cura_interp)), function(i) {
  data.frame(
    ID = dados_teste_modelo$id_paciente[i],
    TIME = tempos_evento_teste,
    PRED_S = as.numeric(S_cura_interp[, i]),
    stringsAsFactors = FALSE
  )
}))

dir.create("../../../ajuste-modelos/cox/dados/saida", recursive = TRUE, showWarnings = FALSE)
arquivo_pred_cura <- "../../../ajuste-modelos/cox/dados/saida/predicao_cura.csv"
write.csv(pred_event_cura, arquivo_pred_cura, row.names = FALSE)

In [ ]:
# Funcao para calcular o CTD
calc_ctd <- function(outcomes_df, pred_event_df) {
  surv_at_event <- pred_event_df %>%
    mutate(
      ID = as.numeric(ID),
      TIME = as.numeric(TIME),
      PRED_S = as.numeric(PRED_S)
    ) %>%
    distinct(ID, TIME, .keep_all = TRUE) %>%
    pivot_wider(names_from = TIME, values_from = PRED_S)

  outcomes <- outcomes_df %>%
    select(ID, FAILTIME, FAILCENS) %>%
    distinct(ID, .keep_all = TRUE) %>%
    arrange(FAILTIME) %>%
    mutate(
      ID = as.numeric(ID),
      FAILTIME = as.numeric(FAILTIME),
      FAILCENS = as.integer(FAILCENS)
    )

  time_cols <- setdiff(names(surv_at_event), "ID")
  time_vals <- as.numeric(time_cols)

  concordante <- 0
  empates <- 0
  pares_comparaveis <- 0

  for (i in seq_len(nrow(outcomes))) {
    if (outcomes$FAILCENS[i] != 1) next

    id_i <- outcomes$ID[i]
    t_i <- outcomes$FAILTIME[i]

    row_i <- surv_at_event[surv_at_event$ID == id_i, , drop = FALSE]
    idx_t <- which(time_vals == t_i)

    if (nrow(row_i) == 0 || length(idx_t) == 0) next

    col_t <- time_cols[idx_t[1]]
    risk_i <- 1 - as.numeric(row_i[[col_t]][1])

    js <- which(outcomes$FAILTIME > t_i)

    for (j in js) {
      id_j <- outcomes$ID[j]
      row_j <- surv_at_event[surv_at_event$ID == id_j, , drop = FALSE]
      if (nrow(row_j) == 0) next

      risk_j <- 1 - as.numeric(row_j[[col_t]][1])
      if (is.na(risk_i) || is.na(risk_j)) next

      pares_comparaveis <- pares_comparaveis + 1
      if (risk_i > risk_j) {
        concordante <- concordante + 1
      } else if (risk_i == risk_j) {
        empates <- empates + 1
      }
    }
  }

  ctd <- if (pares_comparaveis > 0) {
    (concordante + 0.5 * empates) / pares_comparaveis
  } else {
    NA_real_
  }

  c(
    CTD_TESTE = ctd,
    COMPARAVEIS = pares_comparaveis,
    CONCORDANTES = concordante,
    EMPATES = empates
  )
}

# Funcao para calcular CTD dos dois modelos no teste (todos os tempos de evento de intesse nos dados de teste)
calc_ctd_modelos_teste <- function(outcomes_teste, pred_event_cox, pred_event_cura) {
  met_cox <- calc_ctd(outcomes_teste, pred_event_cox)
  met_cura <- calc_ctd(outcomes_teste, pred_event_cura)

  bind_rows(
    data.frame(
      MODELO = "cox",
      CTD_TESTE = as.numeric(met_cox["CTD_TESTE"]),
      COMPARAVEIS = as.numeric(met_cox["COMPARAVEIS"]),
      CONCORDANTES = as.numeric(met_cox["CONCORDANTES"]),
      EMPATES = as.numeric(met_cox["EMPATES"]),
      stringsAsFactors = FALSE
    ),
    data.frame(
      MODELO = "cura",
      CTD_TESTE = as.numeric(met_cura["CTD_TESTE"]),
      COMPARAVEIS = as.numeric(met_cura["COMPARAVEIS"]),
      CONCORDANTES = as.numeric(met_cura["CONCORDANTES"]),
      EMPATES = as.numeric(met_cura["EMPATES"]),
      stringsAsFactors = FALSE
    )
  )
}

outcomes_teste <- dados_teste_modelo %>%
  transmute(
    ID = id_paciente,
    FAILTIME = tempo,
    FAILCENS = delta_t
  )

avaliacao_ctd <- calc_ctd_modelos_teste(outcomes_teste, pred_event_cox, pred_event_cura)
print(avaliacao_ctd)

  MODELO       CENARIO CTD_TESTE COMPARAVEIS CONCORDANTES EMPATES
1    cox todos_eventos 0.7169396       32675        22460    1932
2   cura todos_eventos 0.6548891       32675        20633    1531
